# DIMER Notebook: Open-Vocabulary Object Detection
## Grounding DINO Tiny vs OWLv2 Base/16 Ensemble

**Notebook profile:** `TASK-INFERENCE`  
**Pedagogical mode:** `WORKSHOP`  
**DIMER Notebook Specification:** `2.1`  
**Comparison scope:** `MULTI-MODEL`  
**Standalone:** yes  
**Default workflow:** frozen zero-shot inference — no fine-tuning or adapters

This notebook compares two live DIMER open-vocabulary detectors under one measurable contract:

- **Grounding DINO Tiny** — cross-modal DETR-style grounding over a BERT phrase sequence.
- **OWLv2 Base Patch16 Ensemble** — CLIP-style image-patch/text-query alignment.

The default evaluation uses the deterministic **94-image held-out BCCD split** and the same three concepts for both models:

`red blood cell`, `white blood cell`, `platelet`

### Learning objectives

You will:

- distinguish open-vocabulary from fixed-class detection;
- inspect the different text-conditioning mechanisms used by Grounding DINO and OWLv2;
- normalize heterogeneous detector outputs into one `(score, phrase, box)` representation;
- compare phrase-level AP50, AP75 and recall50;
- inspect localization, phrase confusion, duplicate boxes and model disagreements;
- test prompt wording and prompt ordering as experimental variables; and
- export machine-readable results and provenance.

> **Evidence boundary.** BCCD microscope imagery is a domain-shift stress test, not a general open-vocabulary benchmark. Results here are tutorial measurements on one fixed sample.

## How to use this notebook

**Who it is for.** Learners who can run Python cells in Colab/Jupyter and are new to object detection or language-conditioned vision models.

**Runtime.** Use the documented GPU runtime for the two-model comparison.

**How to run it.**
1. Select the documented runtime/accelerator.
2. Choose **Run all** for the canonical path; leave the default settings unchanged on your first pass.
3. Read the explanatory markdown while the notebook runs.
4. Sections marked **Infrastructure** support reproducibility, model acquisition, or orchestration. Run those cells as written; understanding their implementation is not a learning objective.

### Task at a glance

`image + text prompt → open-vocabulary detector → labelled boxes + scores → localization/detection metrics`

### Roadmap

1. Understand the task and its input/output contract.
2. Inspect and validate the built-in data or inputs.
3. Establish the baseline/reference behavior.
4. Run the model or multi-model comparison.
5. Inspect errors, disagreements, robustness, and/or resource tradeoffs.
6. Try one controlled change and explain what changed.
7. Write an evidence-based conclusion; optionally continue with BYOD.

### What successful execution looks like

You should finish with a validated input/sample, the notebook's principal baseline/reference, model outputs and evaluation results, at least one diagnostic or qualitative comparison, and machine-readable results/provenance where supported. Exact values can vary slightly across supported runtimes; focus on the defined metrics and the observed pattern.


## 1. What makes this task open-vocabulary?

A closed-set detector has a detection head whose class vocabulary is fixed during training.

An open-vocabulary detector instead receives the requested concepts at inference time:

`image + text phrases → phrase-labelled boxes`

That does **not** mean unlimited language understanding. These models are designed primarily for short object phrases, not negation, counting instructions, relations, OCR, or general visual reasoning.

The prompt is part of the effective model configuration.

## 2. Architecture at a glance

### Grounding DINO

`image → Swin-T visual features ↔ BERT phrase sequence → multimodal transformer → 900 object queries → boxes + token-grounding scores`

The prompt list is joined into one sequence such as:

`red blood cell. white blood cell. platelet.`

### OWLv2

`image → CLIP ViT-B/16 → 3,600 patch candidates ↔ independent CLIP text queries → boxes + image/text scores`

Each phrase is encoded as its own text query.

The similar external API therefore hides a meaningful internal difference: **shared phrase sequence versus independent text queries**.

## 3. Configuration

The default values form the canonical `Run all` path.

In [ ]:
USE_BYOD = False  # @param {type:"boolean"}
BYOD_DATASET_PATH = ""  # @param {type:"string"}

EVAL_SCORE_FLOOR = 0.05  # @param {type:"number"}
GDINO_BOX_THRESHOLD = 0.40  # @param {type:"number"}
GDINO_TEXT_THRESHOLD = 0.30  # @param {type:"number"}
OWLV2_THRESHOLD = 0.10  # @param {type:"number"}

RUN_PROMPT_EXPERIMENT = True  # @param {type:"boolean"}
PROMPT_EXPERIMENT_IMAGES = 4  # @param {type:"integer"}
RUN_THRESHOLD_SWEEP = False  # @param {type:"boolean"}

OUTPUT_DIR = "outputs/open_vocabulary_detection"

CANONICAL_PROMPTS = ["red blood cell", "white blood cell", "platelet"]
PROMPT_VARIANTS = {
    "canonical": ["red blood cell", "white blood cell", "platelet"],
    "article": ["a red blood cell", "a white blood cell", "a platelet"],
    "abbreviation": ["RBC", "WBC", "platelet"],
}
PROMPT_VARIANT_TO_CANONICAL = {
    "canonical": {
        "red blood cell": "red blood cell",
        "white blood cell": "white blood cell",
        "platelet": "platelet",
    },
    "article": {
        "a red blood cell": "red blood cell",
        "a white blood cell": "white blood cell",
        "a platelet": "platelet",
    },
    "abbreviation": {
        "rbc": "red blood cell",
        "wbc": "white blood cell",
        "platelet": "platelet",
    },
}

if not 0.0 <= EVAL_SCORE_FLOOR <= 1.0:
    raise ValueError("EVAL_SCORE_FLOOR must be in [0,1]")
if not 1 <= PROMPT_EXPERIMENT_IMAGES <= 12:
    raise ValueError("PROMPT_EXPERIMENT_IMAGES must be in 1..12")
print({
    "canonical_prompts": CANONICAL_PROMPTS,
    "eval_score_floor": EVAL_SCORE_FLOOR,
    "prompt_experiment": RUN_PROMPT_EXPERIMENT,
    "threshold_sweep": RUN_THRESHOLD_SWEEP,
})

## 4. Runtime and reproducibility

Both live carriers use the same Python 3.12 / PyTorch / Transformers runtime family. The notebook pins the principal libraries to the carrier-tested versions.

A GPU is strongly recommended for the full 94-image comparison.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import importlib.metadata as importlib_metadata
import subprocess
import sys

PINS = {
    "torch": "2.14.0",
    "torchvision": "0.29.0",
    "torchaudio": "2.11.0",
    "transformers": "4.57.6",
    "safetensors": "0.8.0",
    "numpy": "2.1.3",
    "pillow": "11.3.0",
    "huggingface-hub": "0.36.2",
}

def dist_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None

from packaging.version import Version

def public_version_matches(actual,expected):
    return actual is not None and Version(str(actual)).public==Version(str(expected)).public

before = {k: dist_version(k) for k in PINS}
needed = [f"{k}=={v}" for k, v in PINS.items() if not public_version_matches(before[k],v)]
if needed:
    print("Installing pinned runtime:", needed)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *needed])

after = {k: dist_version(k) for k in PINS}
bad = {k: (after[k], v) for k, v in PINS.items() if not public_version_matches(after[k],v)}
if bad:
    raise RuntimeError(f"Pinned install did not converge: {bad}")

stale = []
for module_name, dist_name in [("torch","torch"), ("torchvision","torchvision"), ("torchaudio","torchaudio"), ("transformers","transformers"), ("numpy","numpy"), ("PIL","pillow"), ("safetensors","safetensors"), ("huggingface_hub","huggingface-hub")]:
    module = sys.modules.get(module_name)
    if module is not None:
        runtime_version = getattr(module, "__version__", None)
        if runtime_version and not public_version_matches(runtime_version,after[dist_name]):
            stale.append((module_name, runtime_version, after[dist_name]))
if stale:
    raise RuntimeError(
        "This host pre-imported packages that were replaced by the pinned install. "
        f"Restart session, then Run all again; record the restart for qualification. Stale: {stale}"
    )

import numpy as np
import torch
import torchvision
import transformers
import huggingface_hub
from PIL import Image, ImageDraw, ImageFont

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

RUNTIME = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "numpy": np.__version__,
    "pillow": importlib_metadata.version("pillow"),
    "cuda_available": torch.cuda.is_available(),
    "device": DEVICE,
}
if torch.cuda.is_available():
    RUNTIME["gpu_name"] = torch.cuda.get_device_name(0)
    RUNTIME["gpu_total_memory_bytes"] = torch.cuda.get_device_properties(0).total_memory
print(RUNTIME)

## 5. Immutable model provenance

Both checkpoints are SafeTensors snapshots pinned to immutable Hugging Face revisions. The notebook downloads only manifest-listed files, verifies byte size and SHA-256, and loads only from the verified local directory with `trust_remote_code=False`.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import hashlib
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

GDINO_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "grounding-dino-tiny",
  "modelId": "IDEA-Research/grounding-dino-tiny",
  "revision": "a2bb814dd30d776dcf7e30523b00659f4f141c71",
  "files": [
    {
      "path": "README.md",
      "bytes": 2580,
      "sha256": "cf46f74c7b6850f1d5cbe406028324d8798148726016d46a69a365b4a2d3e89f"
    },
    {
      "path": "added_tokens.json",
      "bytes": 82,
      "sha256": "909e96cb32d92ce728a01bc99850cbba26196d74115c17ebeb019275412588f2"
    },
    {
      "path": "config.json",
      "bytes": 1644,
      "sha256": "eec82c5ab66e16df12a9a212e68ac011779927c2536cf9078658e35d85f0c67a"
    },
    {
      "path": "model.safetensors",
      "bytes": 689359096,
      "sha256": "1a2412ef99bd74bcd3c2a246fa1e48581f8889a1300c9051974741314fc042f3"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 457,
      "sha256": "8454179ba95e2ad22947835aad7b45862a601fc0055ab88bf1ee70892d3aea60"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 125,
      "sha256": "b6d346be366a7d1d48332dbc9fdf3bf8960b5d879522b7799ddba59e76237ee3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 711396,
      "sha256": "d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1237,
      "sha256": "d40ab645b68211910b9170d22433d43186a6ec8ee6fd10ba170524b25bf4fb56"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 690308125
}""")
OWLV2_MANIFEST = json.loads(r"""{
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "owlv2-base-patch16-ensemble",
  "modelId": "google/owlv2-base-patch16-ensemble",
  "revision": "cfd3195ba4ea9592eec887ded089f4c08eff231d",
  "files": [
    {
      "path": "README.md",
      "bytes": 4838,
      "sha256": "7c7426bc5ec939a42d1f96fb093031b6263400cceac4129ebb941a0c8c11b9b9"
    },
    {
      "path": "added_tokens.json",
      "bytes": 67,
      "sha256": "e5dc0da35d20111e8ff3fdfc03682beca23d5f94ed74331bce81786b2636a24f"
    },
    {
      "path": "config.json",
      "bytes": 414,
      "sha256": "ba9df8c25a4b8461887dd0a93d9252c9cd84697fe8d49a9d8794ce409af9acb2"
    },
    {
      "path": "merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "model.safetensors",
      "bytes": 619918824,
      "sha256": "e1e130b9e404cf91a75ad45644c1da9d7fa5284085eecc864266a6923efb99e7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 425,
      "sha256": "cf3e396635b797ee1a464e1b2836e98748f8edac19e89aaa2c93b55ac15b0064"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 121,
      "sha256": "d6e2b9cf664efbad2d22998b8d3da986abcbeed3e0825ad33605c9401f9cf73e"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 1100,
      "sha256": "b55cda6198e152ded427c8a9b3faf1cccf27a7fa080697a62f6ff143f511f44f"
    },
    {
      "path": "vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    }
  ],
  "totalBytes": 621510370
}""")

GDINO_DIR = Path("weights/grounding-dino-tiny")
OWLV2_DIR = Path("weights/owlv2-base-patch16-ensemble")
MANIFEST_NAME = "dimer-base-manifest.json"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def stage_and_verify(root, manifest):
    root.mkdir(parents=True, exist_ok=True)
    (root / MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            path.parent.mkdir(parents=True, exist_ok=True)
            hf_hub_download(
                repo_id=manifest["modelId"],
                filename=entry["path"],
                revision=manifest["revision"],
                local_dir=str(root),
            )
    for entry in manifest["files"]:
        path = root / entry["path"]
        if path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{manifest['modelId']} {entry['path']} size mismatch")
        digest = sha256_file(path)
        if digest != entry["sha256"]:
            raise ValueError(f"{manifest['modelId']} {entry['path']} SHA-256 mismatch")
    return {
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest["totalBytes"],
    }

gdino_snapshot = stage_and_verify(GDINO_DIR, GDINO_MANIFEST)
owlv2_snapshot = stage_and_verify(OWLV2_DIR, OWLV2_MANIFEST)
print("Grounding DINO:", gdino_snapshot)
print("OWLv2:", owlv2_snapshot)

## 6. BCCD dataset provenance

The primary sample is the MIT-licensed **BCCD blood-cell detection dataset** from:

`Shenggan/BCCD_Dataset @ d272fb14cdff6e473fafeeeba32aba5f560e9e43`

The DIMER Grounding DINO carrier defines:

- 364 images
- 4,886 bounding boxes
- deterministic split: 220 train / 50 validation / 94 test
- seed: 42
- sample digest: `af9390b9803ac37416f0fcc5b59cc1ef0926b9ea42a0c27510eef3f399e32c08`

This notebook downloads the upstream repository archive at that immutable commit, safely extracts only JPEG images and Pascal-VOC annotations, reconstructs the DIMER records, and verifies the same semantic sample digest over decoded pixels, boxes and phrase labels.

BCCD is a **domain-shift stress test** for open-vocabulary models trained largely on web/everyday imagery.

> **Infrastructure.** This section supports reproducibility and execution. Run the associated setup code as written; understanding its implementation is not a learning objective for this notebook.

In [ ]:
import io
import random
import urllib.request
import zipfile
import xml.etree.ElementTree as ET

BCCD_REPO = "Shenggan/BCCD_Dataset"
BCCD_COMMIT = "d272fb14cdff6e473fafeeeba32aba5f560e9e43"
BCCD_LICENSE = "MIT"
BCCD_ARCHIVE_URL = f"https://github.com/{BCCD_REPO}/archive/{BCCD_COMMIT}.zip"
BCCD_CACHE = Path("weights/bccd")
BCCD_ARCHIVE = BCCD_CACHE / f"{BCCD_COMMIT}.zip"
BCCD_EXPECTED_IMAGES = 364
BCCD_EXPECTED_BOXES = 4_886
BCCD_EXPECTED_IMAGE_BYTES = 7_575_393
BCCD_SAMPLE_DIGEST = "af9390b9803ac37416f0fcc5b59cc1ef0926b9ea42a0c27510eef3f399e32c08"
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 220, "validation": 50, "test": 94}
MAX_EXPANDED_BYTES = 64 * 1024 * 1024

CLASS_PHRASES = {"RBC": "red blood cell", "WBC": "white blood cell", "Platelets": "platelet"}

BCCD_CACHE.mkdir(parents=True, exist_ok=True)
if not BCCD_ARCHIVE.is_file():
    print("Downloading pinned BCCD archive...")
    urllib.request.urlretrieve(BCCD_ARCHIVE_URL, BCCD_ARCHIVE)

images_dir = BCCD_CACHE / "JPEGImages"
annotations_dir = BCCD_CACHE / "Annotations"
images_dir.mkdir(exist_ok=True)
annotations_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(BCCD_ARCHIVE) as zf:
    expanded = 0
    selected = []
    for info in zf.infolist():
        name = info.filename.replace("\\", "/")
        parts = [p for p in name.split("/") if p]
        if info.is_dir():
            continue
        if name.startswith("/") or ".." in parts:
            raise ValueError(f"Unsafe archive member: {name}")
        mode = (info.external_attr >> 16) & 0o170000
        if mode == 0o120000:
            raise ValueError(f"Symlink archive member refused: {name}")
        expanded += info.file_size
        if expanded > MAX_EXPANDED_BYTES:
            raise ValueError("BCCD archive exceeds expanded-size ceiling")
        if "/BCCD/JPEGImages/" in name and name.lower().endswith(".jpg"):
            selected.append((info, images_dir / Path(name).name))
        elif "/BCCD/Annotations/" in name and name.lower().endswith(".xml"):
            selected.append((info, annotations_dir / Path(name).name))
    for info, target in selected:
        if not target.is_file() or target.stat().st_size != info.file_size:
            target.write_bytes(zf.read(info))

image_paths = sorted(images_dir.glob("*.jpg"))
xml_paths = sorted(annotations_dir.glob("*.xml"))

if len(image_paths) != BCCD_EXPECTED_IMAGES:
    raise RuntimeError(f"Expected {BCCD_EXPECTED_IMAGES} BCCD images, found {len(image_paths)}")
if len(xml_paths) != BCCD_EXPECTED_IMAGES:
    raise RuntimeError(f"Expected {BCCD_EXPECTED_IMAGES} annotations, found {len(xml_paths)}")
if sum(p.stat().st_size for p in image_paths) != BCCD_EXPECTED_IMAGE_BYTES:
    raise RuntimeError("Aggregate BCCD image byte count does not match the DIMER sample")

print({
    "archive_sha256_recorded_this_run": sha256_file(BCCD_ARCHIVE),
    "images": len(image_paths),
    "annotations": len(xml_paths),
    "image_bytes": sum(p.stat().st_size for p in image_paths),
})

## 7. Reconstruct and verify the DIMER sample

The dataset digest is order-independent and covers:

- record ID;
- decoded RGB pixels;
- every box;
- every natural-language phrase label.

That provides a semantic equivalence check against the existing DIMER sample even though this notebook retrieves the upstream archive rather than the DIMER pipeline source.

In [ ]:
def image_digest(image):
    rgb = image.convert("RGB")
    payload = f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes()
    return hashlib.sha256(payload).hexdigest()

def dataset_digest(records):
    rows = []
    for r in records:
        objects = sorted(
            (label, [round(float(v), 2) for v in box])
            for box, label in zip(r["boxes"], r["labels"], strict=True)
        )
        rows.append(json.dumps([r["id"], image_digest(r["image"]), objects], separators=(",", ":")))
    return hashlib.sha256("\n".join(sorted(rows)).encode("utf-8")).hexdigest()

def parse_record(image_path, xml_path):
    image = Image.open(image_path)
    image.load()
    image = image.convert("RGB")
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    if size is not None:
        xml_size = (int(size.findtext("width")), int(size.findtext("height")))
        if image.size != xml_size:
            raise ValueError(f"{image_path.stem}: image size {image.size} != annotation {xml_size}")
    boxes, labels = [], []
    for obj in root.findall("object"):
        cls = obj.findtext("name")
        if cls not in CLASS_PHRASES:
            raise ValueError(f"{image_path.stem}: unexpected class {cls!r}")
        b = obj.find("bndbox")
        box = [
            float(b.findtext("xmin")),
            float(b.findtext("ymin")),
            float(b.findtext("xmax")),
            float(b.findtext("ymax")),
        ]
        x0, y0, x1, y1 = box
        if not all(np.isfinite(box)) or not (0 <= x0 < x1 <= image.width and 0 <= y0 < y1 <= image.height):
            raise ValueError(f"{image_path.stem}: invalid box {box}")
        boxes.append(box)
        labels.append(CLASS_PHRASES[cls])
    return {
        "id": f"bccd-{image_path.stem}",
        "image": image,
        "boxes": boxes,
        "labels": labels,
        "source_image_sha256": sha256_file(image_path),
    }

xml_by_stem = {p.stem: p for p in xml_paths}
records = []
for image_path in image_paths:
    if image_path.stem not in xml_by_stem:
        raise RuntimeError(f"Missing annotation for {image_path.name}")
    records.append(parse_record(image_path, xml_by_stem[image_path.stem]))

records.sort(key=lambda r: r["id"])
n_boxes = sum(len(r["boxes"]) for r in records)
if n_boxes != BCCD_EXPECTED_BOXES:
    raise RuntimeError(f"Expected {BCCD_EXPECTED_BOXES} boxes, found {n_boxes}")

sample_digest = dataset_digest(records)
if sample_digest != BCCD_SAMPLE_DIGEST:
    raise RuntimeError(
        f"BCCD semantic sample digest {sample_digest} != pinned DIMER digest {BCCD_SAMPLE_DIGEST}"
    )

pool = list(records)
random.Random(SAMPLE_SEED).shuffle(pool)
splits = {}
cursor = 0
for name, count in SAMPLE_SPLIT.items():
    splits[name] = pool[cursor:cursor+count]
    cursor += count

seen = {}
for split_name, part in splits.items():
    for r in part:
        d = image_digest(r["image"])
        if d in seen:
            raise RuntimeError(f"Decoded image duplicated across {seen[d]} and {split_name}")
        seen[d] = split_name

test_records = splits["test"]
print({
    "sample_digest": sample_digest,
    "images": len(records),
    "boxes": n_boxes,
    "split_sizes": {k: len(v) for k, v in splits.items()},
    "test_boxes": sum(len(r["boxes"]) for r in test_records),
})

## 8. Prompt vocabulary contract

Both models accept at most 16 short phrases. For this notebook the common vocabulary is fixed **before evaluation**:

- `red blood cell`
- `white blood cell`
- `platelet`

OWLv2 has the tighter text ceiling: each phrase is limited to 16 CLIP tokens. Grounding DINO joins all phrases into one BERT sequence limited to 256 tokens.

The canonical prompt vocabulary is not tuned against the 94-image test split.

In [ ]:
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 48

def normalize_prompt_list(prompts):
    if isinstance(prompts, str) or not isinstance(prompts, (list, tuple)):
        raise TypeError("prompts must be a list/tuple of strings")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError("prompt count outside 1..16")
    cleaned = []
    for p in prompts:
        if not isinstance(p, str):
            raise TypeError("every prompt must be str")
        x = " ".join(p.strip().rstrip(".").strip().lower().split())
        if not x or len(x) > MAX_PROMPT_CHARS:
            raise ValueError(f"invalid prompt {p!r}")
        cleaned.append(x)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("prompts must be distinct")
    return cleaned

canonical_prompts = normalize_prompt_list(CANONICAL_PROMPTS)
print(canonical_prompts)

## 9. Common evaluator

Each model is reduced to the same external prediction object:

`(score, phrase, [x0, y0, x1, y1])`

AP is computed per phrase by globally sorting predictions by score, then greedily matching each prediction to the highest-IoU unclaimed ground-truth box of the same phrase in the same image.

Primary metrics:

- AP50
- AP75
- mean AP50 / mean AP75 across the three phrases
- recall50

A supplemental mean AP@[.50:.95] is also calculated for this notebook. It is **not** the full official COCO evaluator.

In [ ]:
import statistics

IOU_THRESHOLDS = tuple(round(0.50 + 0.05*i, 2) for i in range(10))

def box_iou(a, b):
    x0 = max(float(a[0]), float(b[0]))
    y0 = max(float(a[1]), float(b[1]))
    x1 = min(float(a[2]), float(b[2]))
    y1 = min(float(a[3]), float(b[3]))
    iw = max(0.0, x1-x0)
    ih = max(0.0, y1-y0)
    inter = iw*ih
    aa = max(0.0, float(a[2])-float(a[0])) * max(0.0, float(a[3])-float(a[1]))
    bb = max(0.0, float(b[2])-float(b[0])) * max(0.0, float(b[3])-float(b[1]))
    union = aa + bb - inter
    return float(inter/union) if union > 0 else 0.0

def average_precision(predictions, records, phrase, iou_threshold):
    detections = []
    n_gt = 0
    for idx, (preds, record) in enumerate(zip(predictions, records, strict=True)):
        n_gt += sum(label == phrase for label in record["labels"])
        detections.extend(
            (float(score), idx, box)
            for score, label, box in preds
            if label == phrase
        )
    detections.sort(key=lambda x: -x[0])
    matched = {}
    hits = []
    for score, idx, box in detections:
        taken = matched.setdefault(idx, set())
        best_iou, best_j = 0.0, None
        record = records[idx]
        for j, (gt_box, gt_label) in enumerate(zip(record["boxes"], record["labels"], strict=True)):
            if gt_label != phrase or j in taken:
                continue
            iou = box_iou(box, gt_box)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j is not None and best_iou >= iou_threshold:
            taken.add(best_j)
            hits.append(1)
        else:
            hits.append(0)

    if n_gt == 0 or not hits:
        return 0.0, n_gt, int(sum(hits))
    tp = np.cumsum(np.asarray(hits, dtype=np.float64))
    recall = tp / n_gt
    precision = tp / np.arange(1, len(hits)+1)
    mr = np.concatenate([[0.0], recall, [1.0]])
    mp = np.concatenate([[0.0], precision, [0.0]])
    for k in range(len(mp)-2, -1, -1):
        mp[k] = max(mp[k], mp[k+1])
    steps = np.where(mr[1:] != mr[:-1])[0]
    ap = float(np.sum((mr[steps+1]-mr[steps]) * mp[steps+1]))
    return ap, n_gt, int(tp[-1])

def detection_metrics(predictions, records, prompts):
    if len(predictions) != len(records) or not predictions:
        raise ValueError("predictions/records length mismatch")
    per_phrase = {}
    aps_all = []
    for phrase in prompts:
        ap50, n_gt, matched = average_precision(predictions, records, phrase, 0.50)
        ap75, _, _ = average_precision(predictions, records, phrase, 0.75)
        aps = [average_precision(predictions, records, phrase, t)[0] for t in IOU_THRESHOLDS]
        aps_all.append(float(np.mean(aps)))
        per_phrase[phrase] = {
            "n": n_gt,
            "ap50": ap50,
            "ap75": ap75,
            "map50_95": float(np.mean(aps)),
            "recall50": (matched/n_gt) if n_gt else 0.0,
            "n_predictions": sum(
                1 for preds in predictions for _s, label, _b in preds if label == phrase
            ),
        }
    total_gt = sum(v["n"] for v in per_phrase.values())
    return {
        "n_images": len(records),
        "n_boxes": total_gt,
        "map50": float(np.mean([v["ap50"] for v in per_phrase.values()])),
        "map75": float(np.mean([v["ap75"] for v in per_phrase.values()])),
        "map50_95": float(np.mean(aps_all)),
        "recall50": (
            sum(v["recall50"]*v["n"] for v in per_phrase.values())/total_gt
            if total_gt else 0.0
        ),
        "n_predictions": sum(len(x) for x in predictions),
        "per_phrase": per_phrase,
    }

def majority_phrase(train):
    counts = {}
    for r in train:
        for label in r["labels"]:
            counts[label] = counts.get(label, 0) + 1
    return max(counts, key=lambda k: (counts[k], k))

def grid_prior_baseline(train, records, prompts):
    widths, heights = [], []
    for r in train:
        for x0,y0,x1,y1 in r["boxes"]:
            widths.append(x1-x0)
            heights.append(y1-y0)
    mean_w, mean_h = float(np.mean(widths)), float(np.mean(heights))
    phrase = majority_phrase(train)
    predictions = []
    for r in records:
        width, height = r["image"].size
        boxes = []
        y = 0.0
        while y + mean_h <= height + 1e-6:
            x = 0.0
            while x + mean_w <= width + 1e-6:
                boxes.append((1.0, phrase, [x,y,min(x+mean_w,width),min(y+mean_h,height)]))
                x += mean_w
            y += mean_h
        predictions.append(boxes)
    return predictions, detection_metrics(predictions, records, prompts)

grid_predictions, grid_metrics = grid_prior_baseline(splits["train"], test_records, canonical_prompts)
print("Grid-prior baseline:", grid_metrics)

## 10. Predetermined prompt experiments

Prompt experiments use the first four held-out images by stable `image_id`, selected **before looking at predictions**.

Three formulations are tested:

1. canonical: `red blood cell`, `white blood cell`, `platelet`
2. article phrasing: `a red blood cell`, `a white blood cell`, `a platelet`
3. abbreviations: `RBC`, `WBC`, `platelet`

A second experiment reorders the canonical phrase list.

Variant labels are mapped back to canonical concepts **only for analysis**; the text actually given to the model remains unchanged.

In [ ]:
prompt_experiment_records = sorted(test_records, key=lambda r: r["id"])[:PROMPT_EXPERIMENT_IMAGES]
REORDERED_PROMPTS = ["platelet", "red blood cell", "white blood cell"]
print("Prompt experiment images:", [r["id"] for r in prompt_experiment_records])

## 11. Grounding DINO — frozen inference

For quantitative evaluation the notebook follows the DIMER carrier's single-phrase-per-box contract:

- run all 900 object queries;
- reduce token logits within each phrase;
- choose the best phrase per query;
- retain candidates at `EVAL_SCORE_FLOOR`.

This differs from the native visualization path, which additionally applies separate box/text thresholds.

In [ ]:
import gc
import time
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor

GDINO_ID = GDINO_MANIFEST["modelId"]
GDINO_REVISION = GDINO_MANIFEST["revision"]
GDINO_SPECIAL_TOKEN_IDS = (101, 102, 1012, 1029, 0)
GDINO_MAX_TEXT_TOKENS = 256

def gdino_prompt_text(prompts):
    p = normalize_prompt_list(prompts)
    return " ".join(f"{x}." for x in p), p

def phrase_token_groups(input_ids):
    groups, current = [], []
    for position, token in enumerate(input_ids):
        if int(token) in GDINO_SPECIAL_TOKEN_IDS:
            if current:
                groups.append(current)
                current = []
        else:
            current.append(position)
    if current:
        groups.append(current)
    return groups

t0 = time.perf_counter()
gdino_processor = AutoProcessor.from_pretrained(
    str(GDINO_DIR),
    revision=GDINO_REVISION,
    local_files_only=True,
    trust_remote_code=False,
)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    str(GDINO_DIR),
    revision=GDINO_REVISION,
    local_files_only=True,
    trust_remote_code=False,
).to(DEVICE).eval()
for p in gdino_model.parameters():
    p.requires_grad_(False)
p = None  # Release the final parameter alias before the model cleanup stage.
gdino_load_seconds = time.perf_counter() - t0
gdino_parameter_count = sum(p.numel() for p in gdino_model.parameters())

def gdino_predict(image, prompts, score_floor=EVAL_SCORE_FLOOR):
    text, vocab = gdino_prompt_text(prompts)
    inputs = gdino_processor(images=image.convert("RGB"), text=text, return_tensors="pt")
    if int(inputs["input_ids"].shape[1]) > GDINO_MAX_TEXT_TOKENS:
        raise ValueError("Grounding DINO combined prompt exceeds 256 tokens")
    groups = phrase_token_groups(inputs["input_ids"][0].tolist())
    if len(groups) != len(vocab):
        raise RuntimeError(f"Grounding DINO token groups {len(groups)} != phrases {len(vocab)}")
    model_inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        outputs = gdino_model(**model_inputs)
    probs = outputs.logits[0].sigmoid().cpu()
    phrase_scores = torch.stack(
        [probs[:, group].max(dim=1).values for group in groups], dim=1
    )
    score, index = phrase_scores.max(dim=1)
    boxes = outputs.pred_boxes[0].detach().cpu()
    width, height = image.size
    xyxy = torch.stack([
        (boxes[:,0] - boxes[:,2]/2) * width,
        (boxes[:,1] - boxes[:,3]/2) * height,
        (boxes[:,0] + boxes[:,2]/2) * width,
        (boxes[:,1] + boxes[:,3]/2) * height,
    ], dim=1)
    xyxy[:,0::2].clamp_(0, float(width))
    xyxy[:,1::2].clamp_(0, float(height))
    keep = score >= float(score_floor)
    out = [
        (float(s), vocab[int(k)], [float(v) for v in b])
        for s,k,b in zip(score[keep], index[keep], xyxy[keep], strict=True)
    ]
    out.sort(key=lambda x: -x[0])
    return out

def gdino_native_visual(image, prompts):
    text, vocab = gdino_prompt_text(prompts)
    inputs = gdino_processor(images=image.convert("RGB"), text=text, return_tensors="pt")
    model_inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        outputs = gdino_model(**model_inputs)
    result = gdino_processor.post_process_grounded_object_detection(
        outputs,
        inputs["input_ids"],
        threshold=GDINO_BOX_THRESHOLD,
        text_threshold=GDINO_TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]
    labels = result.get("text_labels") or result.get("labels")
    return [
        (float(score), str(label).strip().rstrip(".").lower(), [float(v) for v in box.tolist()])
        for box, score, label in zip(result["boxes"], result["scores"], labels, strict=True)
    ]

# Validate token ceiling before the full run.
probe_inputs = gdino_processor(
    images=test_records[0]["image"],
    text=gdino_prompt_text(canonical_prompts)[0],
    return_tensors="pt",
)
if int(probe_inputs["input_ids"].shape[1]) > GDINO_MAX_TEXT_TOKENS:
    raise RuntimeError("Canonical Grounding DINO prompt exceeds token ceiling")

# Warm-up (excluded from timings).
_ = gdino_predict(test_records[0]["image"], canonical_prompts)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
gdino_predictions = []
gdino_times = []

for i, record in enumerate(test_records):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    preds = gdino_predict(record["image"], canonical_prompts)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    gdino_times.append(time.perf_counter() - start)
    gdino_predictions.append(preds)
    if (i+1) % 20 == 0:
        print(f"Grounding DINO: {i+1}/{len(test_records)}")

gdino_peak_gpu = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
gdino_metrics = detection_metrics(gdino_predictions, test_records, canonical_prompts)

# Native-threshold visual predictions for deterministic examples.
gdino_visual_predictions = {
    r["id"]: gdino_native_visual(r["image"], canonical_prompts)
    for r in prompt_experiment_records
}

# Prompt-variant and prompt-order observations while the model is resident.
gdino_prompt_results = []
if RUN_PROMPT_EXPERIMENT:
    for variant_name, variant_prompts in PROMPT_VARIANTS.items():
        alias = {k.lower(): v for k,v in PROMPT_VARIANT_TO_CANONICAL[variant_name].items()}
        for record in prompt_experiment_records:
            preds = gdino_predict(record["image"], variant_prompts)
            normalized = [(s, alias[label.lower()], b) for s,label,b in preds if label.lower() in alias]
            gdino_prompt_results.append((variant_name, record["id"], normalized))
    for record in prompt_experiment_records[:1]:
        canonical_order = gdino_predict(record["image"], canonical_prompts)
        reordered = gdino_predict(record["image"], REORDERED_PROMPTS)
        gdino_prompt_results.append(("order_canonical", record["id"], canonical_order))
        gdino_prompt_results.append(("order_reordered", record["id"], reordered))

gdino_threshold_sweep = []
if RUN_THRESHOLD_SWEEP:
    for floor in (0.03, 0.05, 0.10, 0.20):
        preds = [gdino_predict(r["image"], canonical_prompts, floor) for r in prompt_experiment_records]
        gdino_threshold_sweep.append((floor, preds))

print("Grounding DINO metrics:", gdino_metrics)
print({
    "parameters": gdino_parameter_count,
    "load_seconds": gdino_load_seconds,
    "mean_inference_seconds": float(np.mean(gdino_times)),
    "peak_gpu_memory_bytes": gdino_peak_gpu,
})

## 12. Release Grounding DINO before loading OWLv2

The comparison retains only normalized predictions and experiment outputs. The first model is removed before the second is loaded so both large checkpoints do not need to reside in accelerator memory simultaneously.

In [ ]:
del gdino_model
del gdino_processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Grounding DINO released.")

## 13. OWLv2 — frozen inference

OWLv2 independently embeds each phrase and scores 3,600 image-patch candidates against them. The processor's grounded-object-detection post-processing selects the best text query for each patch, filters by the score floor, and maps boxes back to original image pixels.

There is no NMS in the canonical path.

In [ ]:
from transformers import Owlv2ForObjectDetection, Owlv2Processor

OWLV2_ID = OWLV2_MANIFEST["modelId"]
OWLV2_REVISION = OWLV2_MANIFEST["revision"]
OWLV2_MAX_TOKENS = 16

t0 = time.perf_counter()
owlv2_processor = Owlv2Processor.from_pretrained(
    str(OWLV2_DIR),
    revision=OWLV2_REVISION,
    local_files_only=True,
    trust_remote_code=False,
)
owlv2_model = Owlv2ForObjectDetection.from_pretrained(
    str(OWLV2_DIR),
    revision=OWLV2_REVISION,
    local_files_only=True,
    trust_remote_code=False,
).to(DEVICE).eval()
for p in owlv2_model.parameters():
    p.requires_grad_(False)
p = None  # Release the final parameter alias before the model cleanup stage.
owlv2_load_seconds = time.perf_counter() - t0
owlv2_parameter_count = sum(p.numel() for p in owlv2_model.parameters())

def owlv2_validate_tokens(prompts):
    queries = normalize_prompt_list(prompts)
    encoded = owlv2_processor.tokenizer(
        queries,
        padding=False,
        truncation=False,
        add_special_tokens=True,
    )
    lengths = [len(x) for x in encoded["input_ids"]]
    if any(n > OWLV2_MAX_TOKENS for n in lengths):
        raise ValueError(f"OWLv2 phrase exceeds {OWLV2_MAX_TOKENS} tokens: {list(zip(queries,lengths))}")
    return queries

def owlv2_predict(image, prompts, threshold=EVAL_SCORE_FLOOR):
    queries = owlv2_validate_tokens(prompts)
    inputs = owlv2_processor(text=[queries], images=image.convert("RGB"), return_tensors="pt")
    model_inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        outputs = owlv2_model(**model_inputs)
    result = owlv2_processor.post_process_grounded_object_detection(
        outputs,
        threshold=float(threshold),
        target_sizes=[image.size[::-1]],
        text_labels=[queries],
    )[0]
    out = [
        (float(score), str(label).strip().rstrip(".").lower(), [float(v) for v in box.tolist()])
        for box, label, score in zip(
            result["boxes"], result["text_labels"], result["scores"], strict=True
        )
    ]
    out.sort(key=lambda x: -x[0])
    if len(out) > 3600:
        raise RuntimeError("OWLv2 returned more than 3,600 patch candidates")
    return out

owlv2_validate_tokens(canonical_prompts)

# Warm-up.
_ = owlv2_predict(test_records[0]["image"], canonical_prompts)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
owlv2_predictions = []
owlv2_times = []

for i, record in enumerate(test_records):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    preds = owlv2_predict(record["image"], canonical_prompts)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    owlv2_times.append(time.perf_counter() - start)
    owlv2_predictions.append(preds)
    if (i+1) % 20 == 0:
        print(f"OWLv2: {i+1}/{len(test_records)}")

owlv2_peak_gpu = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else None
owlv2_metrics = detection_metrics(owlv2_predictions, test_records, canonical_prompts)

owlv2_visual_predictions = {
    r["id"]: owlv2_predict(r["image"], canonical_prompts, OWLV2_THRESHOLD)
    for r in prompt_experiment_records
}

owlv2_prompt_results = []
if RUN_PROMPT_EXPERIMENT:
    for variant_name, variant_prompts in PROMPT_VARIANTS.items():
        alias = {k.lower(): v for k,v in PROMPT_VARIANT_TO_CANONICAL[variant_name].items()}
        for record in prompt_experiment_records:
            preds = owlv2_predict(record["image"], variant_prompts)
            normalized = [(s, alias[label.lower()], b) for s,label,b in preds if label.lower() in alias]
            owlv2_prompt_results.append((variant_name, record["id"], normalized))
    for record in prompt_experiment_records[:1]:
        canonical_order = owlv2_predict(record["image"], canonical_prompts)
        reordered = owlv2_predict(record["image"], REORDERED_PROMPTS)
        owlv2_prompt_results.append(("order_canonical", record["id"], canonical_order))
        owlv2_prompt_results.append(("order_reordered", record["id"], reordered))

owlv2_threshold_sweep = []
if RUN_THRESHOLD_SWEEP:
    for floor in (0.03, 0.05, 0.10, 0.20):
        preds = [owlv2_predict(r["image"], canonical_prompts, floor) for r in prompt_experiment_records]
        owlv2_threshold_sweep.append((floor, preds))

print("OWLv2 metrics:", owlv2_metrics)
print({
    "parameters": owlv2_parameter_count,
    "load_seconds": owlv2_load_seconds,
    "mean_inference_seconds": float(np.mean(owlv2_times)),
    "peak_gpu_memory_bytes": owlv2_peak_gpu,
})

## 14. Common quantitative comparison

Both detectors have now been measured using the same 94 images, the same canonical phrases and the same AP implementation.

The numeric score floor is shared only to retain candidate boxes. It does **not** make the two models' score scales equivalent.

> **What to notice.** Compare the learned-model result with the baseline/reference first, then use the secondary diagnostics to explain the behavior. Do not infer a universal model ranking from one tutorial sample and configuration.

In [ ]:
def pct(x):
    return f"{100*x:.2f}%"

comparison = [
    ("Grid prior", grid_metrics, None, None, None),
    (
        "Grounding DINO Tiny",
        gdino_metrics,
        float(np.mean(gdino_times)),
        GDINO_MANIFEST["totalBytes"],
        gdino_peak_gpu,
    ),
    (
        "OWLv2 Base/16 Ensemble",
        owlv2_metrics,
        float(np.mean(owlv2_times)),
        OWLV2_MANIFEST["totalBytes"],
        owlv2_peak_gpu,
    ),
]

print(f"{'Model':<27} {'mAP50':>9} {'mAP75':>9} {'mAP50-95':>10} {'Recall50':>10} {'Preds':>8}")
print("-"*82)
for name, metrics, *_ in comparison:
    print(
        f"{name:<27} {pct(metrics['map50']):>9} {pct(metrics['map75']):>9} "
        f"{pct(metrics['map50_95']):>10} {pct(metrics['recall50']):>10} "
        f"{metrics['n_predictions']:>8}"
    )

print("\nPer-phrase metrics")
for phrase in canonical_prompts:
    g = gdino_metrics["per_phrase"][phrase]
    o = owlv2_metrics["per_phrase"][phrase]
    print(
        f"{phrase:<18} GT={g['n']:>4} | "
        f"GDINO AP50={g['ap50']:.3f} AP75={g['ap75']:.3f} R50={g['recall50']:.3f} | "
        f"OWLv2 AP50={o['ap50']:.3f} AP75={o['ap75']:.3f} R50={o['recall50']:.3f}"
    )

## 15. Object-level disagreement analysis

Aggregate mAP can hide different failure mechanisms. For every ground-truth object we inspect each model's best overlapping prediction of the **same phrase**.

Categories:

- both detected;
- Grounding-DINO-only;
- OWLv2-only;
- both miss;
- localization disagreement when both succeed but their best IoUs differ by at least 0.20.

This object-level table is diagnostic rather than a standard benchmark metric.

In [ ]:
def best_same_phrase(preds, phrase, gt_box):
    candidates = [(box_iou(box, gt_box), score, box) for score,label,box in preds if label == phrase]
    if not candidates:
        return 0.0, None, None
    iou, score, box = max(candidates, key=lambda x: (x[0], x[1]))
    return float(iou), float(score), box

object_rows = []
score_iou_rows = []
categories = {}

for idx, record in enumerate(test_records):
    for obj_idx, (gt_box, phrase) in enumerate(zip(record["boxes"], record["labels"], strict=True)):
        giou, gscore, gbox = best_same_phrase(gdino_predictions[idx], phrase, gt_box)
        oiou, oscore, obox = best_same_phrase(owlv2_predictions[idx], phrase, gt_box)
        gdet = giou >= 0.50
        odet = oiou >= 0.50
        if gdet and odet:
            category = "localization_disagreement" if abs(giou-oiou) >= 0.20 else "both_detected"
        elif gdet:
            category = "grounding_dino_only"
        elif odet:
            category = "owlv2_only"
        else:
            category = "both_miss"
        categories[category] = categories.get(category, 0) + 1
        row = {
            "image_id": record["id"],
            "object_id": f"{record['id']}-{obj_idx:03d}",
            "prompt": phrase,
            "gt_box": gt_box,
            "grounding_dino_detected": gdet,
            "grounding_dino_score": gscore,
            "grounding_dino_iou": giou,
            "owlv2_detected": odet,
            "owlv2_score": oscore,
            "owlv2_iou": oiou,
            "comparison_category": category,
        }
        object_rows.append(row)
        if gscore is not None:
            score_iou_rows.append(("Grounding DINO Tiny", record["id"], phrase, gscore, giou))
        if oscore is not None:
            score_iou_rows.append(("OWLv2 Base/16 Ensemble", record["id"], phrase, oscore, oiou))

print("Object disagreement categories:", categories)

## 16. Duplicate detections

Neither canonical detector applies NMS. A physical object can therefore attract several same-phrase candidate boxes.

For each ground-truth object, the highest-scoring same-phrase box at IoU ≥ 0.50 is treated as the primary match; additional qualifying boxes are counted as redundant candidates.

AP already penalizes high-ranking duplicates. The counts here simply make the behavior visible.

In [ ]:
def duplicate_summary(predictions, records):
    primary = 0
    redundant = 0
    for preds, record in zip(predictions, records, strict=True):
        for gt_box, phrase in zip(record["boxes"], record["labels"], strict=True):
            matches = [
                (score, box)
                for score,label,box in preds
                if label == phrase and box_iou(box, gt_box) >= 0.50
            ]
            if matches:
                primary += 1
                redundant += max(0, len(matches)-1)
    return {
        "primary_matches": primary,
        "redundant_same_object_boxes": redundant,
        "mean_redundant_per_matched_object": redundant/primary if primary else 0.0,
    }

gdino_duplicates = duplicate_summary(gdino_predictions, test_records)
owlv2_duplicates = duplicate_summary(owlv2_predictions, test_records)
print("Grounding DINO:", gdino_duplicates)
print("OWLv2:", owlv2_duplicates)

## 17. Try it yourself

Before inspecting the prompt experiment output, consider one selected held-out image:

1. Which blood-cell class looks easiest to localize?
2. Which is hardest because of size or frequency?
3. Will both models return the same number of boxes?
4. Can several boxes overlap the same cell without NMS?
5. Should `WBC` necessarily behave exactly like `white blood cell`?

The next cell reveals the measured prompt-variant behavior.

In [ ]:
def prompt_result_summary(model_name, rows):
    summary = []
    record_by_id = {r["id"]: r for r in prompt_experiment_records}
    for variant, image_id, preds in rows:
        if variant.startswith("order_"):
            continue
        record = record_by_id[image_id]
        for phrase in canonical_prompts:
            gt_boxes = [
                b for b,l in zip(record["boxes"], record["labels"], strict=True)
                if l == phrase
            ]
            same = [(s,b) for s,l,b in preds if l == phrase]
            best_score = max([s for s,_b in same], default=None)
            best_iou = 0.0
            for gt in gt_boxes:
                for _s, b in same:
                    best_iou = max(best_iou, box_iou(b, gt))
            summary.append({
                "model": model_name,
                "image_id": image_id,
                "prompt_set": variant,
                "prompt": phrase,
                "n_detections": len(same),
                "best_score": best_score,
                "best_same_class_iou": best_iou,
            })
    return summary

prompt_sensitivity_rows = []
if RUN_PROMPT_EXPERIMENT:
    prompt_sensitivity_rows += prompt_result_summary("Grounding DINO Tiny", gdino_prompt_results)
    prompt_sensitivity_rows += prompt_result_summary("OWLv2 Base/16 Ensemble", owlv2_prompt_results)

    first_id = prompt_experiment_records[0]["id"]
    print("Prompt sensitivity on", first_id)
    for model in ("Grounding DINO Tiny", "OWLv2 Base/16 Ensemble"):
        print("\n", model)
        for row in prompt_sensitivity_rows:
            if row["model"] == model and row["image_id"] == first_id:
                print(
                    f"  {row['prompt_set']:<12} {row['prompt']:<18} "
                    f"n={row['n_detections']:<3} "
                    f"best_score={row['best_score']} best_iou={row['best_same_class_iou']:.3f}"
                )
else:
    print("Prompt experiment disabled.")

## 18. Prompt-order experiment

Grounding DINO receives one concatenated phrase sequence; OWLv2 uses independent text queries. We therefore compare the canonical order with:

`platelet`, `red blood cell`, `white blood cell`

The notebook does **not** assume an effect must occur. It records whatever numerical behavior the current models produce.

In [ ]:
def prediction_signature(preds):
    return [
        (label, round(float(score), 6), tuple(round(float(v), 2) for v in box))
        for score,label,box in preds
    ]

order_results = {}
if RUN_PROMPT_EXPERIMENT:
    for model_name, rows in [
        ("Grounding DINO Tiny", gdino_prompt_results),
        ("OWLv2 Base/16 Ensemble", owlv2_prompt_results),
    ]:
        canonical = next(preds for variant,_id,preds in rows if variant == "order_canonical")
        reordered = next(preds for variant,_id,preds in rows if variant == "order_reordered")
        order_results[model_name] = {
            "canonical_count": len(canonical),
            "reordered_count": len(reordered),
            "rounded_outputs_identical": prediction_signature(canonical) == prediction_signature(reordered),
        }
    print(order_results)
else:
    print("Prompt-order experiment disabled.")

## 19. Optional threshold sweep

A shared numeric floor does not represent equal calibrated confidence. The optional sweep therefore examines **filter sensitivity within each model**, not calibration between models.

When enabled it records candidate count, matched objects, redundant candidates and unmatched predictions on the predetermined prompt-experiment images.

In [ ]:
threshold_sweep_rows = []

def threshold_diagnostics(predictions, records):
    matched_objects = 0
    redundant = 0
    for preds, record in zip(predictions, records, strict=True):
        for gt_box, phrase in zip(record["boxes"], record["labels"], strict=True):
            matches = [
                (s,b) for s,l,b in preds
                if l == phrase and box_iou(b, gt_box) >= 0.50
            ]
            if matches:
                matched_objects += 1
                redundant += max(0, len(matches)-1)
    return {
        "detections": sum(len(x) for x in predictions),
        "matched_objects": matched_objects,
        "redundant_detections": redundant,
    }

if RUN_THRESHOLD_SWEEP:
    for model_name, sweep in [
        ("Grounding DINO Tiny", gdino_threshold_sweep),
        ("OWLv2 Base/16 Ensemble", owlv2_threshold_sweep),
    ]:
        for floor, preds in sweep:
            d = threshold_diagnostics(preds, prompt_experiment_records)
            threshold_sweep_rows.append({
                "model": model_name,
                "score_floor": floor,
                "image_count": len(prompt_experiment_records),
                **d,
            })
    for row in threshold_sweep_rows:
        print(row)
else:
    print("Threshold sweep disabled on canonical Run all path.")

## 20. Confidence versus localization

For every ground-truth object, we retained the best same-phrase box and its model score. This lets us inspect whether high scores coincide with tight localization.

They need not. The score mechanisms also differ between models, so cross-model numeric score comparisons are not calibrated confidence comparisons.

In [ ]:
for model_name in ("Grounding DINO Tiny", "OWLv2 Base/16 Ensemble"):
    rows = [(score,iou) for model,_id,_p,score,iou in score_iou_rows if model == model_name]
    if len(rows) >= 2:
        scores = np.array([x[0] for x in rows], dtype=float)
        ious = np.array([x[1] for x in rows], dtype=float)
        corr = float(np.corrcoef(scores, ious)[0,1]) if scores.std() and ious.std() else float("nan")
        print({
            "model": model_name,
            "pairs": len(rows),
            "mean_score": float(scores.mean()),
            "mean_best_iou": float(ious.mean()),
            "score_iou_pearson": corr,
        })

## 21. Resource and timing comparison

Inference timing excludes package/model download and model loading. Each model received one warm-up inference before the 94 measured calls. CUDA synchronization is used around measured calls when available.

These numbers apply only to the recorded runtime.

In [ ]:
resource_rows = [
    {
        "model": "Grounding DINO Tiny",
        "parameters": gdino_parameter_count,
        "weight_bytes": next(x["bytes"] for x in GDINO_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds": gdino_load_seconds,
        "mean_latency_s": float(np.mean(gdino_times)),
        "median_latency_s": float(np.median(gdino_times)),
        "min_latency_s": float(np.min(gdino_times)),
        "max_latency_s": float(np.max(gdino_times)),
        "peak_gpu_memory_bytes": gdino_peak_gpu,
    },
    {
        "model": "OWLv2 Base/16 Ensemble",
        "parameters": owlv2_parameter_count,
        "weight_bytes": next(x["bytes"] for x in OWLV2_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "load_seconds": owlv2_load_seconds,
        "mean_latency_s": float(np.mean(owlv2_times)),
        "median_latency_s": float(np.median(owlv2_times)),
        "min_latency_s": float(np.min(owlv2_times)),
        "max_latency_s": float(np.max(owlv2_times)),
        "peak_gpu_memory_bytes": owlv2_peak_gpu,
    },
]
for row in resource_rows:
    print(row)

## 22. Qualitative comparison panels

For the predetermined prompt-experiment images the notebook exports:

`Ground Truth | Grounding DINO | OWLv2`

The displayed model boxes use the native readable visualization thresholds:

- Grounding DINO: box 0.40 / text 0.30
- OWLv2: 0.10

Visualizations supplement rather than replace the quantitative output.

In [ ]:
def draw_boxes(image, items, title, gt=False):
    canvas = image.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    for item in items:
        if gt:
            label, box = item
            text = label
        else:
            score, label, box = item
            text = f"{label} {score:.2f}"
        x0,y0,x1,y1 = [float(v) for v in box]
        draw.rectangle([x0,y0,x1,y1], width=2)
        draw.text((x0+2, max(0,y0-12)), text)
    draw.rectangle([0,0,canvas.width,22], fill="white")
    draw.text((6,5), title, fill="black")
    return canvas

def concat_horizontal(images):
    h = max(im.height for im in images)
    w = sum(im.width for im in images)
    out = Image.new("RGB", (w,h), "white")
    x = 0
    for im in images:
        out.paste(im, (x,0))
        x += im.width
    return out

example_dir = Path(OUTPUT_DIR) / "examples"
example_dir.mkdir(parents=True, exist_ok=True)
for record in prompt_experiment_records:
    gt_items = list(zip(record["labels"], record["boxes"], strict=True))
    panels = [
        draw_boxes(record["image"], gt_items, "Ground Truth", gt=True),
        draw_boxes(record["image"], gdino_visual_predictions[record["id"]], "Grounding DINO"),
        draw_boxes(record["image"], owlv2_visual_predictions[record["id"]], "OWLv2"),
    ]
    panel = concat_horizontal(panels)
    path = example_dir / f"{record['id']}_comparison.png"
    panel.save(path)
    print(path)

## 23. Machine-readable exports

The notebook writes:

- `ground_truth.csv`
- `detections.csv`
- `model_metrics.csv`
- `phrase_metrics.csv`
- `object_analysis.csv`
- `prompt_sensitivity.csv`
- `score_iou.csv`
- optional `threshold_sweep.csv`
- `metrics.json`
- `provenance.json`
- deterministic comparison images

In [ ]:
import csv
from datetime import datetime, timezone

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

def write_csv(path, rows, fieldnames):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for row in rows:
            w.writerow({k: row.get(k) for k in fieldnames})

gt_rows = []
for record in test_records:
    for j,(box,label) in enumerate(zip(record["boxes"], record["labels"], strict=True)):
        gt_rows.append({
            "image_id": record["id"],
            "object_id": f"{record['id']}-{j:03d}",
            "prompt": label,
            "x0": box[0], "y0": box[1], "x1": box[2], "y1": box[3],
        })
write_csv(out_dir/"ground_truth.csv", gt_rows, ["image_id","object_id","prompt","x0","y0","x1","y1"])

det_rows = []
for model_name, predictions in [
    ("Grounding DINO Tiny", gdino_predictions),
    ("OWLv2 Base/16 Ensemble", owlv2_predictions),
]:
    for record,preds in zip(test_records,predictions,strict=True):
        for score,label,box in preds:
            det_rows.append({
                "model": model_name, "image_id": record["id"], "prompt": label, "score": score,
                "x0": box[0], "y0": box[1], "x1": box[2], "y1": box[3],
            })
write_csv(out_dir/"detections.csv", det_rows, ["model","image_id","prompt","score","x0","y0","x1","y1"])

model_metric_rows = []
for model_name, metrics, timings, manifest, params, load_s, peak in [
    ("Grounding DINO Tiny", gdino_metrics, gdino_times, GDINO_MANIFEST, gdino_parameter_count, gdino_load_seconds, gdino_peak_gpu),
    ("OWLv2 Base/16 Ensemble", owlv2_metrics, owlv2_times, OWLV2_MANIFEST, owlv2_parameter_count, owlv2_load_seconds, owlv2_peak_gpu),
]:
    model_metric_rows.append({
        "model": model_name,
        "map50": metrics["map50"],
        "map75": metrics["map75"],
        "map50_95": metrics["map50_95"],
        "recall50": metrics["recall50"],
        "n_predictions": metrics["n_predictions"],
        "mean_latency_s": float(np.mean(timings)),
        "median_latency_s": float(np.median(timings)),
        "load_seconds": load_s,
        "parameter_count": params,
        "weight_bytes": next(x["bytes"] for x in manifest["files"] if x["path"]=="model.safetensors"),
        "peak_gpu_memory_bytes": peak,
    })
write_csv(
    out_dir/"model_metrics.csv", model_metric_rows,
    ["model","map50","map75","map50_95","recall50","n_predictions","mean_latency_s","median_latency_s",
     "load_seconds","parameter_count","weight_bytes","peak_gpu_memory_bytes"]
)

phrase_rows = []
for model_name, metrics in [
    ("Grounding DINO Tiny", gdino_metrics),
    ("OWLv2 Base/16 Ensemble", owlv2_metrics),
]:
    for phrase, m in metrics["per_phrase"].items():
        phrase_rows.append({"model":model_name, "prompt":phrase, **m})
write_csv(
    out_dir/"phrase_metrics.csv", phrase_rows,
    ["model","prompt","n","ap50","ap75","map50_95","recall50","n_predictions"]
)

object_export = []
for row in object_rows:
    x = dict(row)
    x["gt_box"] = json.dumps(x["gt_box"])
    object_export.append(x)
write_csv(
    out_dir/"object_analysis.csv", object_export,
    ["image_id","object_id","prompt","gt_box","grounding_dino_detected","grounding_dino_score",
     "grounding_dino_iou","owlv2_detected","owlv2_score","owlv2_iou","comparison_category"]
)

write_csv(
    out_dir/"prompt_sensitivity.csv", prompt_sensitivity_rows,
    ["model","image_id","prompt_set","prompt","n_detections","best_score","best_same_class_iou"]
)

score_rows = [
    {"model":m,"image_id":i,"prompt":p,"score":s,"iou":iou}
    for m,i,p,s,iou in score_iou_rows
]
write_csv(out_dir/"score_iou.csv", score_rows, ["model","image_id","prompt","score","iou"])

if threshold_sweep_rows:
    write_csv(
        out_dir/"threshold_sweep.csv", threshold_sweep_rows,
        ["model","score_floor","image_count","detections","matched_objects","redundant_detections"]
    )

metrics_export = {
    "grid_prior": grid_metrics,
    "grounding_dino": gdino_metrics,
    "owlv2": owlv2_metrics,
    "duplicates": {
        "grounding_dino": gdino_duplicates,
        "owlv2": owlv2_duplicates,
    },
    "object_categories": categories,
    "prompt_order": order_results,
    "timing": {
        "grounding_dino_seconds": gdino_times,
        "owlv2_seconds": owlv2_times,
    },
}
(out_dir/"metrics.json").write_text(json.dumps(metrics_export, indent=2), encoding="utf-8")

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_spec": "2.1",
    "profile": "TASK-INFERENCE",
    "pedagogical_mode": "WORKSHOP",
    "comparison_scope": "MULTI-MODEL",
    "standalone": True,
    "grounding_dino": {
        "model_id": GDINO_ID,
        "revision": GDINO_REVISION,
        "model_safetensors_sha256": next(x["sha256"] for x in GDINO_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "parameter_count": gdino_parameter_count,
        "preprocessing": "native Grounding DINO processor; aspect-preserving resize (short edge 800 / long edge <=1333)",
    },
    "owlv2": {
        "model_id": OWLV2_ID,
        "revision": OWLV2_REVISION,
        "model_safetensors_sha256": next(x["sha256"] for x in OWLV2_MANIFEST["files"] if x["path"]=="model.safetensors"),
        "parameter_count": owlv2_parameter_count,
        "preprocessing": "native OWLv2 processor; square padding then 960x960 CLIP-normalized input",
    },
    "bccd": {
        "repository": BCCD_REPO,
        "commit": BCCD_COMMIT,
        "license": BCCD_LICENSE,
        "images": len(records),
        "boxes": n_boxes,
        "split_seed": SAMPLE_SEED,
        "split_sizes": {k:len(v) for k,v in splits.items()},
        "sample_digest": sample_digest,
        "archive_sha256_recorded_this_run": sha256_file(BCCD_ARCHIVE),
    },
    "canonical_prompts": canonical_prompts,
    "eval_score_floor": EVAL_SCORE_FLOOR,
    "native_visual_thresholds": {
        "grounding_dino_box": GDINO_BOX_THRESHOLD,
        "grounding_dino_text": GDINO_TEXT_THRESHOLD,
        "owlv2": OWLV2_THRESHOLD,
    },
    "prompt_variants": PROMPT_VARIANTS,
    "runtime": RUNTIME,
    "resource_rows": resource_rows,
}
(out_dir/"provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")

print("Exports:")
for p in sorted(out_dir.iterdir()):
    print(" ", p)

## 24. Bring Your Own Data (optional)

The canonical workflow above is complete. BYOD is disabled by default.

Expected labelled dataset structure:

```text
dataset/
  images/
  boxes.csv
```

`boxes.csv` columns:

`image_id,filename,label,x0,y0,x1,y1`

Constraints:

- 8–200 images;
- 1–16 distinct phrases;
- ≤300 boxes per image;
- image sides 16–4096 px;
- phrases ≤48 characters;
- each OWLv2 phrase must fit within 16 text tokens;
- the combined Grounding DINO vocabulary must fit within 256 tokens.

Because this notebook performs no adaptation, supplied labelled data is treated simply as an evaluation sample. It is not called an independent test set unless you supplied it as one.

### Privacy

Do not upload confidential, restricted, sensitive, personal, regulated, or otherwise unauthorized imagery to a hosted notebook runtime. The standalone path does not send your images to DIMER services.

All IDs must map to one file, every image needs annotations, and duplicate pixels/annotations are rejected to avoid reweighting metrics. Total decoded pixels are capped at 100 million. Both token contracts are checked before model loading. Each model is loaded and released sequentially, including on failure. Separate `byod/run-*` exports contain evaluation JSON, prediction JSON/CSV, input digests, model identities and runtime provenance. This path remains pending real-model supported-runtime qualification.


In [ ]:
# Optional BYOD uses the same frozen local inference helpers; no adaptation.
import csv
import tempfile
import traceback
from pathlib import PurePosixPath

def load_byod_dataset(dataset_path):
    root=Path(dataset_path).resolve(); image_dir=(root/'images').resolve(); table=root/'boxes.csv'
    if not image_dir.is_dir() or not table.is_file():
        raise FileNotFoundError('BYOD_DATASET_PATH must contain images/ and boxes.csv')
    with table.open(encoding='utf-8-sig',newline='') as stream:
        reader=csv.DictReader(stream)
        required={'image_id','filename','label','x0','y0','x1','y1'}
        if not required<=set(reader.fieldnames or []): raise ValueError('Missing boxes.csv columns')
        rows=list(reader)
    if not rows or len(rows)>200*300: raise ValueError('Empty or oversized annotations')
    grouped={}; files={}; pixels=set(); total_pixels=0
    for row in rows:
        rid=row['image_id'].strip(); filename=row['filename'].strip()
        rel=PurePosixPath(filename)
        if not rid or not filename or rel.is_absolute() or '..' in rel.parts or '\\' in filename or ':' in filename:
            raise ValueError('Invalid image ID or unsafe filename')
        path=(image_dir/filename).resolve()
        if image_dir not in path.parents or not path.is_file(): raise ValueError('Missing image or path outside images/')
        label=normalize_prompt_list([row['label']])[0]
        if path in files and files[path]!=rid: raise ValueError('One file has multiple image IDs')
        files[path]=rid
        if rid not in grouped:
            with Image.open(path) as source:
                if not (16<=min(source.size) and max(source.size)<=4096): raise ValueError('Image sides outside 16..4096')
                total_pixels+=source.width*source.height
                if total_pixels>100_000_000: raise ValueError('BYOD exceeds 100 million decoded pixels')
                image=source.convert('RGB')
            digest=image_digest(image)
            if digest in pixels: raise ValueError('Duplicate image pixels would reweight evaluation')
            pixels.add(digest)
            grouped[rid]={'id':rid,'filename':filename,'image':image,'boxes':[],'labels':[],
                          'file_sha256':sha256_file(path),'pixel_sha256':digest}
        record=grouped[rid]
        if record['filename']!=filename: raise ValueError('One image ID maps to multiple files')
        box=[float(row[k]) for k in ('x0','y0','x1','y1')]
        if not all(np.isfinite(box)) or not (0<=box[0]<box[2]<=record['image'].width and 0<=box[1]<box[3]<=record['image'].height):
            raise ValueError('Nonfinite or out-of-bounds box')
        if any(box==old and label==lab for old,lab in zip(record['boxes'],record['labels'],strict=True)):
            raise ValueError('Duplicate annotation')
        record['boxes'].append(box); record['labels'].append(label)
        if len(record['boxes'])>300: raise ValueError('More than 300 boxes per image')
    records=list(grouped.values())
    if not 8<=len(records)<=200: raise ValueError('BYOD requires 8..200 distinct images')
    available={p.resolve() for p in image_dir.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}}
    if available!=set(files): raise ValueError('Every image must have annotations; extra images cannot be silently ignored')
    prompts=normalize_prompt_list(sorted({label for r in records for label in r['labels']}))
    return records,prompts,sha256_file(table)

def release_byod_models():
    for name in ('gdino_model','gdino_processor','owlv2_model','owlv2_processor','gp','gm','op','om','p'):
        globals().pop(name,None)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def predict_byod(records,prompts):
    global gdino_processor,gdino_model,owlv2_processor,owlv2_model
    release_byod_models()
    try:
        # Both token contracts are checked before loading either model.
        gdino_processor=AutoProcessor.from_pretrained(str(GDINO_DIR),revision=GDINO_REVISION,local_files_only=True,trust_remote_code=False)
        owlv2_processor=Owlv2Processor.from_pretrained(str(OWLV2_DIR),revision=OWLV2_REVISION,local_files_only=True,trust_remote_code=False)
        text,vocab=gdino_prompt_text(prompts)
        ids=gdino_processor.tokenizer(text,add_special_tokens=True,truncation=False)['input_ids']
        if len(ids)>GDINO_MAX_TEXT_TOKENS or len(phrase_token_groups(ids))!=len(vocab):
            raise ValueError('Grounding DINO vocabulary exceeds token limit or phrase boundaries')
        owlv2_validate_tokens(prompts)
        gdino_model=AutoModelForZeroShotObjectDetection.from_pretrained(str(GDINO_DIR),revision=GDINO_REVISION,local_files_only=True,trust_remote_code=False).to(DEVICE).eval()
        gd=[gdino_predict(r['image'],prompts) for r in records]
        del gdino_model,gdino_processor
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        owlv2_model=Owlv2ForObjectDetection.from_pretrained(str(OWLV2_DIR),revision=OWLV2_REVISION,local_files_only=True,trust_remote_code=False).to(DEVICE).eval()
        owl=[owlv2_predict(r['image'],prompts) for r in records]
        return gd,owl
    except Exception as exc:
        traceback.clear_frames(exc.__traceback__)
        raise
    finally:
        release_byod_models()

def export_byod(records,prompts,annotation_sha,gd,owl):
    parent=Path(OUTPUT_DIR)/'byod'; parent.mkdir(parents=True,exist_ok=True)
    target=Path(tempfile.mkdtemp(prefix='run-',dir=parent))
    result={'prompts':prompts,'images':len(records),'evaluation_scope':'supplied labelled sample; no adaptation',
            'grounding_dino':detection_metrics(gd,records,prompts),'owlv2':detection_metrics(owl,records,prompts)}
    manifest={'annotation_sha256':annotation_sha,'records':[{k:v for k,v in r.items() if k!='image'} for r in records],
              'runtime':RUNTIME,'score_floor':EVAL_SCORE_FLOOR,
              'grounding_dino':GDINO_MANIFEST,'owlv2':OWLV2_MANIFEST,'prompts':prompts}
    (target/'input_provenance.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    (target/'evaluation.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    predictions={'image_ids':[r['id'] for r in records],'grounding_dino':gd,'owlv2':owl}
    (target/'predictions.json').write_text(json.dumps(predictions,indent=2),encoding='utf-8')
    with (target/'detections.csv').open('w',encoding='utf-8',newline='') as stream:
        writer=csv.writer(stream);writer.writerow(['model','image_id','score','phrase','x0','y0','x1','y1'])
        for model,preds in [('grounding_dino',gd),('owlv2',owl)]:
            for record,detections in zip(records,preds,strict=True):
                for score,phrase,box in detections:writer.writerow([model,record['id'],score,phrase,*box])
    return result,target

byod_result=None
if USE_BYOD:
    byod_records,byod_prompts,byod_annotation_sha=load_byod_dataset(BYOD_DATASET_PATH)
    gd_byod,owl_byod=predict_byod(byod_records,byod_prompts)
    byod_result,byod_output=export_byod(byod_records,byod_prompts,byod_annotation_sha,gd_byod,owl_byod)
    print(byod_result); print('Separate BYOD exports:',byod_output)
else:
    print('BYOD disabled on the canonical Run all path.')


## 25. Interpretation and limitations

### Open vocabulary is still a constrained interface

Text conditioning lets an operator change the requested vocabulary without retraining, but these detectors are not general visual-language reasoning engines.

### Prompt wording changes the task

`white blood cell` and `WBC` produce different text representations. A prompt policy is therefore part of a deployed open-vocabulary detector.

### The models ground language differently

Grounding DINO performs cross-modal fusion over one BERT phrase sequence. OWLv2 aligns image patches against independent CLIP text-query embeddings. Similar output schemas do not imply identical internal semantics.

### BCCD is a domain-shift stress test

Microscope images differ markedly from common web photographs. Weak results may reflect visual-domain shift, small-object difficulty, phrase representation, localization limits, class imbalance, or several factors at once.

### No NMS means duplicates are visible

Both canonical paths intentionally preserve their no-NMS behavior. This matters if detections are later counted or converted into pseudo-labels.

### Scores are not cross-model probabilities

A Grounding DINO score of 0.5 and an OWLv2 score of 0.5 do not represent equivalent confidence. The underlying score mechanisms differ and neither is calibrated for this deployment domain.

### Prompt experiments are exploratory

The canonical phrases were fixed before test evaluation. Prompt variants demonstrate sensitivity; they are not used to tune the canonical evaluation against the held-out split.

## 26. Terminal summary

The final cell verifies that required exports exist and prints only measurements from the current execution.

In [ ]:
required_outputs = [
    out_dir/"ground_truth.csv",
    out_dir/"detections.csv",
    out_dir/"model_metrics.csv",
    out_dir/"phrase_metrics.csv",
    out_dir/"object_analysis.csv",
    out_dir/"prompt_sensitivity.csv",
    out_dir/"score_iou.csv",
    out_dir/"metrics.json",
    out_dir/"provenance.json",
]
missing = [str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise RuntimeError(f"Required outputs missing: {missing}")

print("DIMER Open-Vocabulary Detection Workshop")
print("-"*43)
print("Dataset:")
print(f"  BCCD test images: {len(test_records)}")
print(f"  Prompt vocabulary: {len(canonical_prompts)}")
print(f"  Ground-truth objects: {sum(len(r['boxes']) for r in test_records)}")
print()
print("Grounding DINO Tiny")
print(f"  mAP50: {gdino_metrics['map50']:.4f}")
print(f"  mAP75: {gdino_metrics['map75']:.4f}")
print(f"  mAP50-95: {gdino_metrics['map50_95']:.4f}")
print(f"  recall50: {gdino_metrics['recall50']:.4f}")
print(f"  predictions: {gdino_metrics['n_predictions']}")
print(f"  mean inference time: {np.mean(gdino_times):.4f} s")
print()
print("OWLv2 Base/16 Ensemble")
print(f"  mAP50: {owlv2_metrics['map50']:.4f}")
print(f"  mAP75: {owlv2_metrics['map75']:.4f}")
print(f"  mAP50-95: {owlv2_metrics['map50_95']:.4f}")
print(f"  recall50: {owlv2_metrics['recall50']:.4f}")
print(f"  predictions: {owlv2_metrics['n_predictions']}")
print(f"  mean inference time: {np.mean(owlv2_times):.4f} s")
print()
print(f"Prompt sensitivity images: {len(prompt_experiment_records) if RUN_PROMPT_EXPERIMENT else 0}")
print(f"Outputs: {out_dir}/")

## Try it yourself — one controlled change

**Predict → change one variable → run → observe → explain.** In a separate notebook copy and fresh runtime, keep all defaults except `RUN_THRESHOLD_SWEEP=True`. Predict whether raising the score floor from 0.05 to 0.20 reduces recall and duplicate detections. Run all, compare the existing threshold-diagnostic rows for those two floors on the same `prompt_experiment_records`, and explain one model difference using the printed values. Do not select a new canonical threshold from this held-out diagnostic. Keep the original canonical outputs and the exploratory exports in separate directories. Both models have been released or may be released by BYOD after the completed run; rerunning their inference cells alone is not a valid continuation without reconstruction.


## Self-paced checkpoint

Before opening the sample interpretation, answer:

1. What did the model/system receive as input, and what did it produce?
2. Which baseline/reference tells you whether the learned model added value?
3. What failure mode or tradeoff matters most here?
4. What additional evidence would you want before transferring the result to a new domain?

<details>
<summary><b>Show a sample interpretation</b></summary>

Prompt wording is part of the task definition. Higher confidence does not automatically mean better localization, and scores from Grounding DINO and OWLv2 are not calibrated on a shared probability scale. The BCCD sample is also a domain-shift stress test rather than a universal detection benchmark.

Use the outputs from **your run** when writing your final answer; small numeric differences across supported runtimes are possible.

</details>


## Write an evidence-based conclusion

1. **State the question** tested by this notebook.
2. **Report the primary result** against the relevant baseline/reference.
3. **Add supporting evidence** from a secondary metric, error pattern, disagreement, or qualitative diagnostic.
4. **Account for cost/complexity** when it materially affects the comparison.
5. **State the limits** of the data, split, model revision, and configuration.

Compare both detectors against the common evaluator and reference conditions, report the main localization/detection metric, identify a prompt or duplicate-detection failure mode, and state the threshold/domain limitations that constrain the result.


# Troubleshooting

| What you see | Likely cause | What to do |
|---|---|---|
| Accelerator unavailable or execution is unexpectedly slow | The runtime does not match the documented resource envelope | Select the documented accelerator/runtime, start a fresh session, and run top-to-bottom. |
| Package/version or stale-module error | Incompatible libraries were already imported in the hosted kernel | Start a fresh runtime and choose **Run all** before importing extra packages. Do not bypass version checks. |
| Model/sample digest or byte-size check fails | Download is incomplete or upstream bytes differ from the pinned artifact | Remove the affected runtime cache/download and rerun. Do not disable the integrity check. |
| Out-of-memory or runtime restart | Too many large models/intermediates are resident | Use the default tier, follow explicit unload/release steps, and avoid combining optional heavy branches. |
| BYOD validation fails | Input does not satisfy the documented schema, shape, labels, or limits | Follow the validation message, correct the indicated field/format, then rerun the BYOD branch. |
| Your numbers differ slightly | Supported hardware/library execution can introduce small numerical variation | Verify the split, model revision, metric definition, and qualitative pattern before treating the difference as substantive. |


# Glossary

| Term | Meaning in this notebook |
|---|---|
| **Open-vocabulary detection** | Detecting objects named by text rather than only a fixed trained class list. |
| **Prompt** | The text query that specifies what object concept to detect. |
| **Bounding box** | A rectangle locating a detected object in an image. |
| **IoU** | Intersection over Union; overlap between a predicted and reference box. |
| **Precision** | Among returned detections, the fraction that are correct under the evaluator. |
| **Recall** | Among reference objects, the fraction detected under the evaluator. |